In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class myState(TypedDict):
    label: str
    order: int
    response: str
    openai_response: str
    gemini_response: str


# -------------------------
# LINEAR NODES (safe)
# -------------------------
def Image_Recognition_agent(state: myState) -> myState:
    state["label"] = "Image_Recognition"
    state["order"] = 1
    state["response"] = "Image Recognized"
    return state


def Recommendation_Agent(state: myState) -> myState:
    state["label"] = "Recommendation_Agent"
    state["order"] = 2
    state["response"] = "Recommendation Generated"
    return state


# -------------------------
# PARALLEL NODES (IMPORTANT FIX)
# DO NOT TOUCH label/order/response here
# -------------------------
def OpenAI_agent(state: myState) -> myState:
    state["openai_response"] = "OpenAI Response Generated"
    return state


def GEMENI_AI_Agent(state: myState) -> myState:
    state["gemini_response"] = "Gemini Response Generated"
    return state


# -------------------------
# GRAPH
# -------------------------
builder = StateGraph(myState)

builder.add_node("Image_Recognition", Image_Recognition_agent)
builder.add_node("Recommendation_Agent", Recommendation_Agent)
builder.add_node("Open_AI_Agent", OpenAI_agent)
builder.add_node("GEMENI_AI_Agent", GEMENI_AI_Agent)

builder.add_edge(START, "Image_Recognition")
builder.add_edge("Image_Recognition", "Recommendation_Agent")

# 🔥 PARALLEL BRANCH
builder.add_edge("Recommendation_Agent", "Open_AI_Agent")
builder.add_edge("Recommendation_Agent", "GEMENI_AI_Agent")

builder.add_edge("Open_AI_Agent", END)
builder.add_edge("GEMENI_AI_Agent", END)

graph = builder.compile()

print(graph.get_graph().draw_ascii())

initial_state = {
    "label": "",
    "order": 0,
    "response": "",
    "openai_response": "",
    "gemini_response": ""
}

result = graph.invoke(initial_state)

print("\nFINAL RESULT:")
print(result)

                  +-----------+                 
                  | __start__ |                 
                  +-----------+                 
                        *                       
                        *                       
                        *                       
              +-------------------+             
              | Image_Recognition |             
              +-------------------+             
                        *                       
                        *                       
                        *                       
            +----------------------+            
            | Recommendation_Agent |            
            +----------------------+            
                ***           ***               
              **                 **             
            **                     **           
+-----------------+           +---------------+ 
| GEMENI_AI_Agent |           | Open_AI_Agent | 
+-----------------+ 

InvalidUpdateError: At key 'label': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE